<a href="https://colab.research.google.com" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DataWranglerTool — Quickstart Notebook

A robust Python toolkit for **data wrangling, feature engineering, interactive visualisation,
and statistical association analysis**, optimised for Google Colab.

## Features at a Glance

- **Intelligent Data Loading** — CSV, TSV, XLSX, JSON, Parquet; auto null-string handling; smart type inference
- **Comprehensive Profiling** — shape, schema, per-column deep report, missing-value heatmap
- **Automated Cleaning** — 7 null strategies, duplicate removal, IQR outlier handling, outlier flagging
- **Reshaping** — interactive column / row deletion, rename, cast, reorder
- **Feature Engineering** — binning, datetime expansion, ratio columns, custom transforms
- **Encoding & Scaling** — one-hot / ordinal / frequency encoding; min-max / standard / robust / log1p / sqrt scaling
- **Interactive Visualisations** — violin+box+histogram panel, pair plot, auto relationship chart, time series
- **Statistical Insights** — Pearson / Spearman, Cramér's V, unified η² heatmap
- **Snapshots & Export** — named rollback checkpoints, CSV/Excel export, operation log

## 📦 Installation

In [ ]:
!pip install "git+https://github.com/your-org/data-wrangling-tool.git[plotting]" -q

In [ ]:
from data_wrangling import DataWrangler, ChartBuilder
import pandas as pd

## 1️⃣  Data Loading

In [ ]:
wrangler = DataWrangler()

# Option A — interactive Colab upload
# wrangler.upload_data()

# Option B — load from URL (Titanic dataset used here)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
wrangler.load_url(url)

## 2️⃣  Profiling & Inspection

In [ ]:
wrangler.get_profile()

In [ ]:
wrangler.show_schema()

In [ ]:
wrangler.column_report('Age')

In [ ]:
wrangler.get_numeric_summary()

In [ ]:
wrangler.get_categorical_summary()

In [ ]:
wrangler.show_missing_heatmap()

## 3️⃣  Cleaning

In [ ]:
# Snapshot before cleaning so we can roll back
wrangler.snapshot('raw')

In [ ]:
# Impute missing values using median for numeric, mode for categorical
wrangler.fix_nulls(strategy='median')

In [ ]:
wrangler.drop_duplicates()

In [ ]:
# View IQR outlier flags without changing data
flags = wrangler.flag_outliers(columns=['Age', 'Fare'])
flags.sum()

In [ ]:
# Clip outliers to IQR bounds
wrangler.fix_outliers(columns=['Fare'], strategy='clip')

## 4️⃣  Reshaping

In [ ]:
# Drop noisy columns
wrangler.drop_columns(['Cabin', 'Ticket', 'Name'])

In [ ]:
wrangler.rename_columns({'Pclass': 'passenger_class', 'SibSp': 'siblings_spouses'})

In [ ]:
wrangler.cast_columns({'passenger_class': 'category', 'Survived': 'bool'})

## 5️⃣  Feature Engineering

In [ ]:
wrangler.bin_column('Age', bins=[0, 12, 18, 60, 120],
                    labels=['Child', 'Teen', 'Adult', 'Senior'])

In [ ]:
wrangler.add_ratio_column('Fare', 'Age', new_column='fare_per_year')

In [ ]:
import numpy as np
wrangler.apply_transform('Fare', np.log1p, new_column='Fare_log')

## 6️⃣  Encoding & Scaling

In [ ]:
# Get a fully processed ML-ready copy (does NOT change self.df)
ml_df = wrangler.get_encoded_df(numeric_method='robust', categorical_method='onehot')
ml_df.head()

## 7️⃣  Visualisations

In [ ]:
# Three-panel distribution view for numeric columns
wrangler.plot_numeric(['Age', 'Fare'])

In [ ]:
wrangler.plot_categorical(['Sex', 'Embarked', 'passenger_class'])

In [ ]:
# Scatter matrix coloured by survival
wrangler.plot_pair(['Age', 'Fare', 'fare_per_year'], color='Sex')

In [ ]:
# Auto-selects the best chart type based on column dtypes
wrangler.plot_relationship('Sex', 'Fare')

## 8️⃣  Correlation & Association

In [ ]:
wrangler.plot_numeric_heatmap(method='pearson')

In [ ]:
wrangler.plot_categorical_heatmap()

In [ ]:
# Unified heatmap: |Pearson r| for num-num, Cramér's V for cat-cat, η² for num-cat
wrangler.plot_unified_heatmap()

## 9️⃣  Custom Charts with ChartBuilder

In [ ]:
cb = ChartBuilder()

# All available chart methods
pd.DataFrame(cb.get_methods_info()['response'])

In [ ]:
result = cb.plot_bar(wrangler.df, x='Sex', y='Fare', color='passenger_class', barmode='group')
cb.display(result)

In [ ]:
result = cb.plot_pie(wrangler.df, names='Embarked', values='Fare', hole=0.4, title='Fare by Embarkation')
cb.display(result)

In [ ]:
result = cb.plot_histogram(wrangler.df, x='Age', kde=True, title='Age Distribution')
cb.display(result)

In [ ]:
result = cb.plot_heatmap(wrangler.df, index='Sex', columns='passenger_class',
                         values='Fare', agg='mean', title='Mean Fare by Sex and Class')
cb.display(result)

In [ ]:
result = cb.plot_sankey(wrangler.df, source_col='Sex', target_col='Embarked', value_col='Fare')
cb.display(result)

In [ ]:
result = cb.plot_sunburst(wrangler.df, path=['Sex', 'passenger_class'], values='Fare')
cb.display(result)

In [ ]:
result = cb.plot_treemap(wrangler.df, path=['Sex', 'passenger_class'], values='Fare')
cb.display(result)

In [ ]:
result = cb.plot_bubble(wrangler.df, x='Age', y='Fare', size='siblings_spouses', color='Sex')
cb.display(result)

## 🔟  Snapshots, Export & History

In [ ]:
wrangler.snapshot('cleaned')
wrangler.export_csv('titanic_wrangled.csv')

In [ ]:
# Roll back to the raw data if needed
# wrangler.restore('raw')

In [ ]:
# Full operation history
wrangler.get_history()